## LightGBM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
from lightgbm import LGBMRegressor
from skopt import BayesSearchCV
from skopt.space import Real, Integer
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据
df = pd.read_excel('Style Preference Training Set.xlsx')

# 数据预处理
print("数据基本信息:")
print(f"数据集形状: {df.shape}")
print("\n前5行数据:")
print(df.head())

# 检查缺失值
print("\n缺失值检查:")
print(df.isnull().sum())

# 如果有缺失值，使用均值填充
df.fillna(df.mean(numeric_only=True), inplace=True)

# 独热编码
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_features = encoder.fit_transform(df[['款式', '省份']])
encoded_df = pd.DataFrame(encoded_features, 
                         columns=encoder.get_feature_names_out(['款式', '省份']))

# 归一化处理
scaler = MinMaxScaler()
normalized_features = scaler.fit_transform(df[['搜索指数', '资讯指数']])
normalized_df = pd.DataFrame(normalized_features, 
                            columns=['搜索指数_norm', '资讯指数_norm'])

# 合并处理后的数据
processed_df = pd.concat([
    encoded_df,
    normalized_df,
    df[['人口（万人）', '消费支出', '服装偏好']].reset_index(drop=True)
], axis=1)

# 准备特征和目标变量
X = processed_df.drop('服装偏好', axis=1)
y = processed_df['服装偏好']

print(f"\n处理后的特征维度: {X.shape}")

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 定义LightGBM模型
lgbm = LGBMRegressor(random_state=42, verbose=-1)

# 定义贝叶斯搜索的参数空间
param_space = {

    'n_estimators': Integer(100, 1000),
    'max_depth': Integer(3, 20),
    'learning_rate': Real(0.005, 0.2, prior='log-uniform'),
    'num_leaves': Integer(20, 150),
    'min_child_samples': Integer(1, 100),
    'subsample': Real(0.7, 1.0),
    'colsample_bytree': Real(0.7, 1.0),
    'reg_alpha': Real(0, 20),
    'reg_lambda': Real(0, 20)
}

# 贝叶斯参数寻优
bayes_search = BayesSearchCV(
    estimator=lgbm,
    search_spaces=param_space,
    n_iter=30,  # 迭代次数
    cv=5,       # 5折交叉验证
    scoring='neg_mean_squared_error',
    random_state=42,
    verbose=1,
    n_jobs=-1
)

# 进行参数寻优
print("开始贝叶斯参数寻优...")
bayes_search.fit(X_train, y_train)

print("最佳参数:", bayes_search.best_params_)
print("最佳分数 (负MSE):", bayes_search.best_score_)

# 使用最佳参数训练最终模型
best_model = bayes_search.best_estimator_

# 在测试集上进行预测
y_pred = best_model.predict(X_test)

# 计算各种评估指标
def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    # 计算平均绝对百分比误差（处理零值）
    mask = y_true != 0
    if np.any(mask):
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.nan
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Explained Variance': evs,
        'MAPE': mape
    }

# 计算测试集评估指标
test_metrics = calculate_metrics(y_test, y_pred)

print("\n=== 测试集评估结果 ===")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.6f}")

# 五折交叉验证评估
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_metrics = {
    'MSE': [],
    'RMSE': [],
    'MAE': [],
    'R2': [],
    'Explained Variance': []
}

print("\n=== 五折交叉验证结果 ===")
for fold, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # 使用最佳参数训练模型
    model = LGBMRegressor(**bayes_search.best_params_, random_state=42, verbose=-1)
    model.fit(X_train_fold, y_train_fold)
    
    # 预测
    y_pred_fold = model.predict(X_val_fold)
    
    # 计算指标
    fold_metrics = calculate_metrics(y_val_fold, y_pred_fold)
    
    for metric in cv_metrics.keys():
        if metric in fold_metrics:
            cv_metrics[metric].append(fold_metrics[metric])
    
    print(f"Fold {fold}: R2 = {fold_metrics['R2']:.4f}, RMSE = {fold_metrics['RMSE']:.6f}")

# 计算交叉验证的平均指标
print("\n=== 五折交叉验证平均指标 ===")
for metric, values in cv_metrics.items():
    if values: 
        print(f"平均{metric}: {np.mean(values):.6f} (±{np.std(values):.6f})")

# 特征重要性分析
feature_importance = best_model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("\n=== 特征重要性排名（前10） ===")
print(importance_df.head(10))

# 可视化评估结果
def plot_evaluation_results(y_true, y_pred):
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. 预测值 vs 真实值散点图
    axes[0, 0].scatter(y_true, y_pred, alpha=0.6)
    axes[0, 0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    axes[0, 0].set_xlabel('真实值')
    axes[0, 0].set_ylabel('预测值')
    axes[0, 0].set_title('预测值 vs 真实值')
    axes[0, 0].grid(True)
    
    # 2. 残差图
    residuals = y_true - y_pred
    axes[0, 1].scatter(y_pred, residuals, alpha=0.6)
    axes[0, 1].axhline(y=0, color='r', linestyle='--')
    axes[0, 1].set_xlabel('预测值')
    axes[0, 1].set_ylabel('残差')
    axes[0, 1].set_title('残差图')
    axes[0, 1].grid(True)
    
    # 3. 误差分布直方图
    axes[1, 0].hist(residuals, bins=30, alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(x=0, color='r', linestyle='--')
    axes[1, 0].set_xlabel('残差')
    axes[1, 0].set_ylabel('频率')
    axes[1, 0].set_title('残差分布')
    axes[1, 0].grid(True)
    
    # 4. 特征重要性条形图（前10）
    top_features = importance_df.head(10)
    axes[1, 1].barh(range(len(top_features)), top_features['importance'])
    axes[1, 1].set_yticks(range(len(top_features)))
    axes[1, 1].set_yticklabels(top_features['feature'])
    axes[1, 1].set_xlabel('重要性得分')
    axes[1, 1].set_title('Top 10 特征重要性')
    
    plt.tight_layout()
    plt.savefig('model_evaluation_plots.png', dpi=300, bbox_inches='tight')
    plt.show()

# 绘制评估图表
plot_evaluation_results(y_test, y_pred)

# 训练最终模型在所有数据上
final_model = LGBMRegressor(**bayes_search.best_params_, random_state=42, verbose=-1)
final_model.fit(X, y)

print("\n最终模型训练完成！")

# 保存模型和相关处理器
import joblib
joblib.dump(final_model, 'lgbm_clothing_preference_model.pkl')
joblib.dump(encoder, 'onehot_encoder.pkl')
joblib.dump(scaler, 'minmax_scaler.pkl')
joblib.dump(importance_df, 'feature_importance.pkl')

print("模型和相关处理器已保存！")

# 模型评估
print("\n" + "="*50)
print("="*50)
print(f"测试集 R² 分数: {test_metrics['R2']:.4f}")
print(f"测试集 RMSE: {test_metrics['RMSE']:.6f}")
print(f"测试集 MAE: {test_metrics['MAE']:.6f}")
print(f"平均交叉验证 R²: {np.mean(cv_metrics['R2']):.4f}")
print(f"最重要的特征: {importance_df.iloc[0]['feature']} "
      f"(重要性: {importance_df.iloc[0]['importance']:.2f})")

## XGBoost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import KFold, train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import random

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据
df = pd.read_excel('Style Preference Training Set.xlsx')

# 数据预处理
print("数据基本信息:")
print(f"数据集形状: {df.shape}")
print("\n前5行数据:")
print(df.head())

# 检查缺失值
print("\n缺失值检查:")
print(df.isnull().sum())

# 如果有缺失值，进行填充
df.fillna(df.mean(numeric_only=True), inplace=True)

# 独热编码
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_features = encoder.fit_transform(df[['款式', '省份']])
encoded_df = pd.DataFrame(encoded_features, 
                         columns=encoder.get_feature_names_out(['款式', '省份']))

# 归一化处理
scaler = MinMaxScaler()
normalized_features = scaler.fit_transform(df[['搜索指数', '资讯指数']])
normalized_df = pd.DataFrame(normalized_features, 
                            columns=['搜索指数_norm', '资讯指数_norm'])

# 合并处理后的数据
processed_df = pd.concat([
    encoded_df,
    normalized_df,
    df[['人口（万人）', '消费支出', '服装偏好']].reset_index(drop=True)
], axis=1)

# 准备特征和目标变量
X = processed_df.drop('服装偏好', axis=1)
y = processed_df['服装偏好']

print(f"\n处理后的特征维度: {X.shape}")
print(f"目标变量范围: {y.min():.6f} - {y.max():.6f}")

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 计算评估指标函数
def calculate_metrics(y_true, y_pred, model_name=""):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    
    mask = y_true != 0
    if np.any(mask):
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.nan
    
    metrics = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Explained Variance': evs,
        'MAPE': mape
    }
    
    if model_name:
        print(f"\n=== {model_name} 评估结果 ===")
        for metric, value in metrics.items():
            if metric == 'MAPE' and np.isnan(value):
                print(f"{metric}: N/A (零值无法计算)")
            else:
                print(f"{metric}: {value:.6f}")
    
    return metrics

# 随机搜索函数
def custom_random_search(X_train, y_train, n_iter=50, cv=5):
    param_space = {
        'n_estimators': [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000],
        'max_depth': [3, 4, 5, 6, 7, 8, 9, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'gamma': [0, 0.1, 0.2, 0.3, 0.4, 0.5],
        'reg_alpha': [0, 0.1, 0.5, 1, 2],
        'reg_lambda': [0, 0.1, 0.5, 1, 2]
    }
    
    best_score = float('inf')
    best_params = None
    best_model = None
    results = []
    
    print("开始自定义随机搜索参数寻优...")
    
    for i in range(n_iter):
        # 随机选择参数
        params = {}
        for param, values in param_space.items():
            params[param] = random.choice(values)
        
        # 创建模型
        model = XGBRegressor(
            **params,
            random_state=42,
            verbosity=0,
            n_jobs=-1
        )
        
        # 交叉验证
        try:
            scores = cross_val_score(model, X_train, y_train, cv=cv, 
                                   scoring='neg_mean_squared_error', n_jobs=-1)
            mse_score = -np.mean(scores)
            
            results.append({
                'params': params.copy(),
                'mse': mse_score
            })
            
            if mse_score < best_score:
                best_score = mse_score
                best_params = params.copy()
                
            if (i + 1) % 10 == 0:
                print(f"迭代 {i+1}/{n_iter}, 当前最佳MSE: {best_score:.6f}")
                
        except Exception as e:
            print(f"参数组合 {params} 训练失败: {e}")
            continue
    

    if best_params is None:
        print("所有参数组合都失败了")
        best_params = {
            'n_estimators': 100,
            'max_depth': 6,
            'learning_rate': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'gamma': 0,
            'reg_alpha': 0,
            'reg_lambda': 1
        }
        best_score = float('inf')
    
    # 使用最佳参数训练最终模型
    print(f"使用最佳参数训练最终模型: {best_params}")
    best_model = XGBRegressor(**best_params, random_state=42, verbosity=0, n_jobs=-1)
    best_model.fit(X_train, y_train)
    
    # 重新计算最佳MSE
    y_pred = best_model.predict(X_train)
    best_score = mean_squared_error(y_train, y_pred)
    
    print("自定义随机搜索完成!")
    print(f"最佳参数: {best_params}")
    print(f"最佳MSE: {best_score:.6f}")
    
    # 按MSE排序结果
    results.sort(key=lambda x: x['mse'])
    
    return best_model, best_params, best_score, results


# 进行自定义随机搜索参数寻优
xgb_best_model, xgb_best_params, xgb_best_mse, search_results = custom_random_search(
    X_train, y_train, n_iter=50, cv=5
)

# 检查模型是否成功创建
if xgb_best_model is None:
    print("错误: 模型训练失败，使用默认模型")
    xgb_best_model = XGBRegressor(random_state=42, verbosity=0, n_jobs=-1)
    xgb_best_model.fit(X_train, y_train)
    xgb_best_params = xgb_best_model.get_params()

# 在测试集上进行预测
xgb_y_pred = xgb_best_model.predict(X_test)

# 计算测试集评估指标
xgb_test_metrics = calculate_metrics(y_test, xgb_y_pred, "XGBoost测试集")


# 五折交叉验证
print("\n" + "="*60)
print("五折交叉验证结果")
print("="*60)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_metrics = {
    'MSE': [],
    'RMSE': [],
    'MAE': [],
    'R2': [],
    'Explained Variance': []
}

print("五折交叉验证详细结果:")
for fold, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # 使用最佳参数训练模型
    model = XGBRegressor(**xgb_best_params, random_state=42, verbosity=0, n_jobs=-1)
    model.fit(X_train_fold, y_train_fold)
    
    # 预测
    y_pred_fold = model.predict(X_val_fold)
    
    # 计算指标
    fold_metrics = calculate_metrics(y_val_fold, y_pred_fold)
    
    for metric in cv_metrics.keys():
        if metric in fold_metrics:
            cv_metrics[metric].append(fold_metrics[metric])
    
    print(f"Fold {fold}: R² = {fold_metrics['R2']:.4f}, RMSE = {fold_metrics['RMSE']:.6f}")

# 计算交叉验证的平均指标
print("\n五折交叉验证平均指标:")
for metric, values in cv_metrics.items():
    if values:
        print(f"平均{metric}: {np.mean(values):.6f} (±{np.std(values):.6f})")


# 特征重要性分析
print("\n" + "="*60)
print("特征重要性分析")
print("="*60)

feature_importance = xgb_best_model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("特征重要性排名（前10）:")
print(importance_df.head(10))

# =============================================================================
# 可视化结果
# =============================================================================
print("\n生成可视化图表...")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. 预测值 vs 真实值散点图
axes[0, 0].scatter(y_test, xgb_y_pred, alpha=0.6, color='red')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('真实值')
axes[0, 0].set_ylabel('预测值')
axes[0, 0].set_title('XGBoost预测值 vs 真实值')
axes[0, 0].grid(True)

# 2. 残差图
residuals = y_test - xgb_y_pred
axes[0, 1].scatter(xgb_y_pred, residuals, alpha=0.6, color='green')
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('预测值')
axes[0, 1].set_ylabel('残差')
axes[0, 1].set_title('残差图')
axes[0, 1].grid(True)

# 3. 误差分布直方图
axes[1, 0].hist(residuals, bins=20, alpha=0.7, edgecolor='black', color='orange')
axes[1, 0].axvline(x=0, color='r', linestyle='--')
axes[1, 0].set_xlabel('残差')
axes[1, 0].set_ylabel('频率')
axes[1, 0].set_title('残差分布')
axes[1, 0].grid(True)

# 4. 特征重要性条形图（前10）
top_features = importance_df.head(10)
axes[1, 1].barh(range(len(top_features)), top_features['importance'], color='purple')
axes[1, 1].set_yticks(range(len(top_features)))
axes[1, 1].set_yticklabels(top_features['feature'])
axes[1, 1].set_xlabel('重要性得分')
axes[1, 1].set_title('Top 10 特征重要性')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('xgboost_evaluation_plots.png', dpi=300, bbox_inches='tight')
plt.show()

# 学习曲线分析
print("\n生成学习曲线...")

# 绘制学习曲线
train_sizes = np.linspace(0.1, 1.0, 10)
train_scores = []
test_scores = []

for size in train_sizes:
    n_samples = int(len(X_train) * size)
    X_subset = X_train.iloc[:n_samples]
    y_subset = y_train.iloc[:n_samples]
    
    model = XGBRegressor(**xgb_best_params, random_state=42, verbosity=0)
    model.fit(X_subset, y_subset)
    
    train_score = model.score(X_subset, y_subset)
    test_score = model.score(X_test, y_test)
    
    train_scores.append(train_score)
    test_scores.append(test_score)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores, 'o-', color='blue', label='训练集得分')
plt.plot(train_sizes, test_scores, 'o-', color='red', label='测试集得分')
plt.xlabel('训练集比例')
plt.ylabel('R²分数')
plt.title('XGBoost学习曲线')
plt.legend()
plt.grid(True)
plt.savefig('xgboost_learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()

# 最终模型训练和保存
print("\n" + "="*60)
print("最终模型训练和保存")
print("="*60)

# 使用所有数据训练最终模型
final_model = XGBRegressor(**xgb_best_params, random_state=42, verbosity=0, n_jobs=-1)
final_model.fit(X, y)

print("最终XGBoost模型训练完成！")

# 保存模型和相关处理器
import joblib
joblib.dump(final_model, 'xgboost_final_model.pkl')
joblib.dump(encoder, 'onehot_encoder.pkl')
joblib.dump(scaler, 'minmax_scaler.pkl')
joblib.dump(importance_df, 'feature_importance.pkl')

# 保存最佳参数和搜索结果
pd.DataFrame([xgb_best_params]).to_csv('xgboost_best_params.csv', index=False)
if search_results:
    search_results_df = pd.DataFrame(search_results)
    search_results_df.to_csv('xgboost_search_results.csv', index=False)

print("模型和相关处理器已保存！")

# 性能评估
print("\n" + "="*60)
print("="*60)
print(f"测试集 R² 分数: {xgb_test_metrics['R2']:.4f}")
print(f"测试集 RMSE: {xgb_test_metrics['RMSE']:.6f}")
print(f"测试集 MAE: {xgb_test_metrics['MAE']:.6f}")
print(f"平均交叉验证 R²: {np.mean(cv_metrics['R2']):.4f}")
print(f"最重要的特征: {importance_df.iloc[0]['feature']} "
      f"(重要性: {importance_df.iloc[0]['importance']:.4f})")

print(f"\n最佳参数配置:")
for param, value in xgb_best_params.items():
    print(f"  {param}: {value}")

print("\nXGBoost模型训练和评估完成！")

## Random Forest

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import KFold, train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
from sklearn.ensemble import RandomForestRegressor
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据
df = pd.read_excel('Style Preference Training Set.xlsx')

# 数据预处理
print("数据基本信息:")
print(f"数据集形状: {df.shape}")
print("\n前5行数据:")
print(df.head())

# 检查缺失值
print("\n缺失值检查:")
print(df.isnull().sum())

# 如果有缺失值进行填充
df.fillna(df.mean(numeric_only=True), inplace=True)

# 独热编码
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_features = encoder.fit_transform(df[['款式', '省份']])
encoded_df = pd.DataFrame(encoded_features, 
                         columns=encoder.get_feature_names_out(['款式', '省份']))

# 归一化处理
scaler = MinMaxScaler()
normalized_features = scaler.fit_transform(df[['搜索指数', '资讯指数']])
normalized_df = pd.DataFrame(normalized_features, 
                            columns=['搜索指数_norm', '资讯指数_norm'])

# 合并处理后的数据
processed_df = pd.concat([
    encoded_df,
    normalized_df,
    df[['人口（万人）', '消费支出', '服装偏好']].reset_index(drop=True)
], axis=1)

# 准备特征和目标变量
X = processed_df.drop('服装偏好', axis=1)
y = processed_df['服装偏好']

print(f"\n处理后的特征维度: {X.shape}")
print(f"目标变量范围: {y.min():.6f} - {y.max():.6f}")

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 计算评估指标函数
def calculate_metrics(y_true, y_pred, model_name=""):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    
    mask = y_true != 0
    if np.any(mask):
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.nan
    
    metrics = {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Explained Variance': evs,
        'MAPE': mape
    }
    
    if model_name:
        print(f"\n=== {model_name} 评估结果 ===")
        for metric, value in metrics.items():
            print(f"{metric}: {value:.6f}")
    
    return metrics

# 贝叶斯优化函数
def random_forest_bayesian_optimization(X_train, y_train, n_iter=50, cv=5):
    # 定义参数空间
    dimensions = [
        Integer(100, 500, name='n_estimators'),
        Integer(5, 30, name='max_depth'),
        Integer(2, 20, name='min_samples_split'),
        Integer(1, 10, name='min_samples_leaf'),
        Real(0.1, 1.0, name='max_features'),
        Categorical([True, False], name='bootstrap')
    ]
    
    param_names = ['n_estimators', 'max_depth', 'min_samples_split', 
                  'min_samples_leaf', 'max_features', 'bootstrap']
    
    @use_named_args(dimensions=dimensions)
    def objective(n_estimators, max_depth, min_samples_split, 
                 min_samples_leaf, max_features, bootstrap):
        
        model = RandomForestRegressor(
            n_estimators=int(n_estimators),
            max_depth=int(max_depth) if max_depth > 0 else None,
            min_samples_split=int(min_samples_split),
            min_samples_leaf=int(min_samples_leaf),
            max_features=max_features,
            bootstrap=bootstrap,
            random_state=42,
            n_jobs=-1
        )
        
        # 使用负MSE作为评分
        scores = cross_val_score(model, X_train, y_train, cv=cv, 
                               scoring='neg_mean_squared_error', n_jobs=-1)
        return -np.mean(scores)  # 返回正MSE
    
    print("开始随机森林贝叶斯参数寻优...")
    result = gp_minimize(objective, dimensions, n_calls=n_iter, 
                        random_state=42, verbose=True)
    
    # 获取最佳参数
    best_params = {}
    for i, param_name in enumerate(param_names):
        best_params[param_name] = result.x[i]
        # 对整数参数进行转换
        if param_name in ['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf']:
            best_params[param_name] = int(best_params[param_name])
    
    print("贝叶斯优化完成!")
    print(f"最佳参数: {best_params}")
    print(f"最佳MSE: {result.fun:.6f}")
    
    # 使用最佳参数训练最终模型
    best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
    best_model.fit(X_train, y_train)
    
    return best_model, best_params, result.fun


# 随机森林模型训练和评估
print("\n" + "="*60)
print("随机森林模型训练和评估")
print("="*60)

# 进行贝叶斯参数寻优
rf_best_model, rf_best_params, rf_best_mse = random_forest_bayesian_optimization(
    X_train, y_train, n_iter=50, cv=5
)

# 在测试集上进行预测
rf_y_pred = rf_best_model.predict(X_test)

# 计算测试集评估指标
rf_test_metrics = calculate_metrics(y_test, rf_y_pred, "随机森林测试集")


# 五折交叉验证
print("\n" + "="*60)
print("五折交叉验证结果")
print("="*60)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_metrics = {
    'MSE': [],
    'RMSE': [],
    'MAE': [],
    'R2': [],
    'Explained Variance': []
}

print("五折交叉验证详细结果:")
for fold, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # 使用最佳参数训练模型
    model = RandomForestRegressor(**rf_best_params, random_state=42, n_jobs=-1)
    model.fit(X_train_fold, y_train_fold)
    
    # 预测
    y_pred_fold = model.predict(X_val_fold)
    
    # 计算指标
    fold_metrics = calculate_metrics(y_val_fold, y_pred_fold)
    
    for metric in cv_metrics.keys():
        if metric in fold_metrics:
            cv_metrics[metric].append(fold_metrics[metric])
    
    print(f"Fold {fold}: R² = {fold_metrics['R2']:.4f}, RMSE = {fold_metrics['RMSE']:.6f}")

# 计算交叉验证的平均指标
print("\n五折交叉验证平均指标:")
for metric, values in cv_metrics.items():
    if values:
        print(f"平均{metric}: {np.mean(values):.6f} (±{np.std(values):.6f})")


# 特征重要性分析
print("\n" + "="*60)
print("特征重要性分析")
print("="*60)

feature_importance = rf_best_model.feature_importances_
feature_names = X.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("特征重要性排名（前10）:")
print(importance_df.head(10))


# 可视化结果
print("\n生成可视化图表...")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. 预测值 vs 真实值散点图
axes[0, 0].scatter(y_test, rf_y_pred, alpha=0.6, color='blue')
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('真实值')
axes[0, 0].set_ylabel('预测值')
axes[0, 0].set_title('随机森林预测值 vs 真实值')
axes[0, 0].grid(True)

# 2. 残差图
residuals = y_test - rf_y_pred
axes[0, 1].scatter(rf_y_pred, residuals, alpha=0.6, color='green')
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('预测值')
axes[0, 1].set_ylabel('残差')
axes[0, 1].set_title('残差图')
axes[0, 1].grid(True)

# 3. 误差分布直方图
axes[1, 0].hist(residuals, bins=20, alpha=0.7, edgecolor='black', color='orange')
axes[1, 0].axvline(x=0, color='r', linestyle='--')
axes[1, 0].set_xlabel('残差')
axes[1, 0].set_ylabel('频率')
axes[1, 0].set_title('残差分布')
axes[1, 0].grid(True)

# 4. 特征重要性条形图（前10）
top_features = importance_df.head(10)
axes[1, 1].barh(range(len(top_features)), top_features['importance'], color='purple')
axes[1, 1].set_yticks(range(len(top_features)))
axes[1, 1].set_yticklabels(top_features['feature'])
axes[1, 1].set_xlabel('重要性得分')
axes[1, 1].set_title('Top 10 特征重要性')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('random_forest_evaluation_plots.png', dpi=300, bbox_inches='tight')
plt.show()


# 最终模型训练和保存
print("\n" + "="*60)
print("最终模型训练和保存")
print("="*60)

# 使用所有数据训练最终模型
final_model = RandomForestRegressor(**rf_best_params, random_state=42, n_jobs=-1)
final_model.fit(X, y)

print("最终模型训练完成！")

# 保存模型和相关处理器
import joblib
joblib.dump(final_model, 'random_forest_final_model.pkl')
joblib.dump(encoder, 'onehot_encoder.pkl')
joblib.dump(scaler, 'minmax_scaler.pkl')
joblib.dump(importance_df, 'feature_importance.pkl')

print("模型和相关处理器已保存！")

# 性能总结
print("\n" + "="*60)
print("="*60)
print(f"测试集 R² 分数: {rf_test_metrics['R2']:.4f}")
print(f"测试集 RMSE: {rf_test_metrics['RMSE']:.6f}")
print(f"测试集 MAE: {rf_test_metrics['MAE']:.6f}")
print(f"平均交叉验证 R²: {np.mean(cv_metrics['R2']):.4f}")
print(f"最重要的特征: {importance_df.iloc[0]['feature']} "
      f"(重要性: {importance_df.iloc[0]['importance']:.4f})")

print("\n随机森林模型训练和评估完成！")

## Use the optimal Random Forest model for prediction

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os

def predict_for_each_file():
    """
    使用训练好的模型对每个文件进行预测
    """
    # 加载训练好的模型和处理器
    try:
        model = joblib.load('random_forest_final_model.pkl')
        encoder = joblib.load('onehot_encoder.pkl')
        scaler = joblib.load('minmax_scaler.pkl')
        print("成功加载模型和处理器")
    except Exception as e:
        print(f"加载模型失败: {e}")
        return
    
    # 处理每个年份的预测表
    for year in range(2024, 2101):
        pred_file = f"Style Preference Prediction Dataset and Results//{year}年预测.xlsx"
        
        if os.path.exists(pred_file):
            try:
                print(f"\n处理 {year} 年预测表...")
                
                # 读取数据
                df = pd.read_excel(pred_file)
                original_shape = df.shape
                print(f"原始数据形状: {original_shape}")
                
                # 检查必要的列是否存在
                required_columns = ['款式', '省份', '搜索指数', '资讯指数', '人口（万人）', '消费支出']
                missing_columns = [col for col in required_columns if col not in df.columns]
                
                if missing_columns:
                    print(f"缺少必要列: {missing_columns}，跳过该文件")
                    continue
                
                original_df = df.copy()
                
                # 数据预处理
                # 1. 独热编码
                encoded_features = encoder.transform(df[['款式', '省份']])
                encoded_df = pd.DataFrame(encoded_features, 
                                         columns=encoder.get_feature_names_out(['款式', '省份']))
                
                # 2. 归一化处理
                normalized_features = scaler.transform(df[['搜索指数', '资讯指数']])
                normalized_df = pd.DataFrame(normalized_features, 
                                           columns=['搜索指数_norm', '资讯指数_norm'])
                
                # 3. 合并处理后的数据
                processed_df = pd.concat([
                    encoded_df,
                    normalized_df,
                    df[['人口（万人）', '消费支出']].reset_index(drop=True)
                ], axis=1)
                
                print(f"处理后的特征维度: {processed_df.shape}")
                
                # 进行预测
                predictions = model.predict(processed_df)
                
                # 将预测结果添加到原始数据中
                df['服装偏好预测'] = predictions
                
                # 保存预测结果
                output_file = f"Style Preference Prediction Dataset and Results//{year}年预测_带预测结果.xlsx"
                df.to_excel(output_file, index=False)
                
                print(f"预测完成！预测结果已保存到: {output_file}")
                print(f"预测值范围: {predictions.min():.6f} - {predictions.max():.6f}")
                print(f"预测值均值: {predictions.mean():.6f}")
                
            except Exception as e:
                print(f"处理 {year} 年预测表时出错: {e}")
        else:
            print(f"预测表不存在: {pred_file}")

def predict_with_detailed_analysis():
    """
    带详细分析的预测版本
    """
    # 加载模型和处理器
    try:
        model = joblib.load('random_forest_final_model.pkl')
        encoder = joblib.load('onehot_encoder.pkl')
        scaler = joblib.load('minmax_scaler.pkl')
        feature_importance = joblib.load('feature_importance.pkl')
        print("成功加载模型、处理器和特征重要性数据")
    except Exception as e:
        print(f"加载失败: {e}")
        return
    
    # 创建结果汇总表
    summary_results = []
    
    for year in range(2024, 2101):
        pred_file = f"Style Preference Prediction Dataset and Results//{year}年预测.xlsx"
        
        if not os.path.exists(pred_file):
            continue
            
        try:
            print(f"\n=== 处理 {year} 年预测表 ===")
            
            # 读取数据
            df = pd.read_excel(pred_file)
            
            # 检查必要列
            required_columns = ['款式', '省份', '搜索指数', '资讯指数', '人口（万人）', '消费支出']
            if not all(col in df.columns for col in required_columns):
                print("缺少必要列，跳过")
                continue
            
            # 数据预处理
            encoded_features = encoder.transform(df[['款式', '省份']])
            encoded_df = pd.DataFrame(encoded_features, 
                                     columns=encoder.get_feature_names_out(['款式', '省份']))
            
            normalized_features = scaler.transform(df[['搜索指数', '资讯指数']])
            normalized_df = pd.DataFrame(normalized_features, 
                                       columns=['搜索指数_norm', '资讯指数_norm'])
            
            processed_df = pd.concat([
                encoded_df,
                normalized_df,
                df[['人口（万人）', '消费支出']].reset_index(drop=True)
            ], axis=1)
            
            # 预测
            predictions = model.predict(processed_df)
            df['服装偏好预测'] = predictions
            
            # 保存结果
            output_file = f"款式偏好预测(最终)//{year}年预测_带预测结果.xlsx"
            df.to_excel(output_file, index=False)
            
            # 统计信息
            stats = {
                '年份': year,
                '记录数': len(df),
                '预测均值': predictions.mean(),
                '预测标准差': predictions.std(),
                '预测最小值': predictions.min(),
                '预测最大值': predictions.max(),
                '预测中位数': np.median(predictions)
            }
            summary_results.append(stats)
            
            print(f"预测完成: {output_file}")
            print(f"统计信息: 均值={predictions.mean():.4f}, 标准差={predictions.std():.4f}")
            
            # 显示每个款式的预测统计
            print("\n各款式预测统计:")
            style_stats = df.groupby('款式')['服装偏好预测'].agg(['mean', 'std', 'count']).round(4)
            print(style_stats)
            
        except Exception as e:
            print(f"错误: {e}")
    
    # 保存汇总结果
    if summary_results:
        summary_df = pd.DataFrame(summary_results)
        summary_file = "预测结果汇总统计.xlsx"
        summary_df.to_excel(summary_file, index=False)
        print(f"\n预测汇总结果已保存到: {summary_file}")
        
        # 显示汇总统计
        print("\n=== 各年份预测汇总 ===")
        print(summary_df)

def predict_specific_files(file_list=None):
    """
    预测特定文件列表
    """
    if file_list is None:
        file_list = [f"Style Preference Prediction Dataset and Results//{year}年预测.xlsx" for year in range(2024, 2101)]
    
    # 加载模型
    try:
        model = joblib.load('random_forest_final_model.pkl')
        encoder = joblib.load('onehot_encoder.pkl')
        scaler = joblib.load('minmax_scaler.pkl')
    except Exception as e:
        print(f"加载模型失败: {e}")
        return
    
    for file_path in file_list:
        if os.path.exists(file_path):
            try:
                print(f"\n处理文件: {file_path}")
                
                df = pd.read_excel(file_path)
                
                # 数据预处理和预测
                encoded_features = encoder.transform(df[['款式', '省份']])
                encoded_df = pd.DataFrame(encoded_features, 
                                         columns=encoder.get_feature_names_out(['款式', '省份']))
                
                normalized_features = scaler.transform(df[['搜索指数', '资讯指数']])
                normalized_df = pd.DataFrame(normalized_features, 
                                           columns=['搜索指数_norm', '资讯指数_norm'])
                
                processed_df = pd.concat([
                    encoded_df,
                    normalized_df,
                    df[['人口（万人）', '消费支出']].reset_index(drop=True)
                ], axis=1)
                
                predictions = model.predict(processed_df)
                df['服装偏好预测'] = predictions
                
                # 保存结果
                base_name = os.path.splitext(file_path)[0]
                output_file = f"{base_name}_带预测结果.xlsx"
                df.to_excel(output_file, index=False)
                
                print(f"预测完成: {output_file}")
                
            except Exception as e:
                print(f"处理失败: {e}")

# 使用示例
if __name__ == "__main__":
    print("开始进行预测...")
    predict_for_each_file()
    
    print("\n所有预测完成！")